# 01 · Data summary, missingness and episodes

Dataset summary (Table 1), station correlation and missing-data map (Figure S2) and the haze / exceedance episode counts per split.

*Estimated time: < 1 min.*

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
import pandas as pd
from IPython.display import Image, display
from airq.env import resolve_paths
from airq.pipelines.data import run
from airq.pipelines.figures import eda

paths = resolve_paths(cfg["paths"])
if not (paths["processed"] / "clean.parquet").exists():
    raise SystemExit("Run notebook 00 first (it builds data/processed/clean.parquet).")
pd.read_csv(paths["reports"] / "tables" / "dataset_summary.csv")


In [ ]:
pdf_png = [p for p in eda(cfg) if p.suffix == ".png"][0]
display(Image(filename=str(pdf_png)))


In [ ]:
rows = []
for split in ("train", "val", "test"):
    ep = pd.read_csv(paths["reports"] / "tables" / f"episodes_{split}.csv")
    rows.append(ep.groupby(["type", "threshold"]).size().rename(split))
pd.concat(rows, axis=1).fillna(0).astype(int)


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
